# 🌸 Tutorial Didático: Aprendizado Federado com Flower e PyTorch

Este notebook é um ambiente **100% autocontido e canônico** para aprender e experimentar com **Aprendizado Federado (Federated Learning)** no dataset CIFAR-10, utilizando o framework [Flower](https://flower.ai/) e PyTorch.

---
### ⚡ Configuração Inicial de GPU:
Para treinar com máxima velocidade na GPU T4 gratuita do Colab:
No menu superior, vá em: **Ambiente de execução (Runtime) -> Alterar tipo de ambiente de execução -> Acelerador: T4 GPU -> Salvar**.

## 1. Verificar Disponibilidade da GPU T4

In [ ]:
!nvidia-smi

## 2. Instalação das Dependências Oficiais
Instala o Flower, PyTorch, Torchvision e a biblioteca de particionamento `flwr-datasets`.

In [ ]:
!pip install -q datasets "flwr-datasets[vision]>=0.5.0" torch torchvision matplotlib numpy flwr
print("✔ Dependências do Flower e PyTorch instaladas com sucesso!")

## 3. Painel de Controle Interativo (Configuração Sem Código)
Ajuste os parâmetros da federação diretamente pelo formulário abaixo e execute a simulação.

> 💡 **Dica de Execução:** Para rodar o fluxo completo de uma vez, clique em **Ambiente de execução -> Executar tudo** (`Ctrl + F9`). Se preferir rodar célula por célula, certifique-se de executar a **Célula 2** antes desta.

In [ ]:
#@title ⚙️ Painel de Controle da Simulação Federada { display-mode: "form" }
#@markdown ### 1. Estratégia de Agregação do Servidor
estrategia_agregacao = "FedAvg" #@param ["FedAvg", "FedMedian"]

#@markdown ---
#@markdown ### 2. Hiperparâmetros da Federação
num_rodadas = 10 #@param {type:"integer"}
num_clientes = 10 #@param {type:"integer"}
epocas_locais = 2 #@param {type:"integer"}
learning_rate = 0.01 #@param {type:"number"}
batch_size = 64 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}

#@markdown ---
#@markdown ### 3. Heterogeneidade dos Dados (Particionamento Dirichlet)
distribuicao_dados = "IID (Homogêneo - Alpha=100.0)" #@param ["IID (Homogêneo - Alpha=100.0)", "Não-IID Suave (Alpha=1.0)", "Não-IID Moderado (Alpha=0.5)", "Não-IID Severo (Alpha=0.1)"]

mapa_alpha = {
    "IID (Homogêneo - Alpha=100.0)": 100.0,
    "Não-IID Suave (Alpha=1.0)": 1.0,
    "Não-IID Moderado (Alpha=0.5)": 0.5,
    "Não-IID Severo (Alpha=0.1)": 0.1,
}
alpha_escolhido = mapa_alpha[distribuicao_dados]

# Verificação de segurança: instala dependências automaticamente caso a Célula 2 não tenha sido executada
try:
    import flwr_datasets
    import flwr
except ImportError:
    print('⏳ Dependências ausentes detectadas. Instalando pacotes no Colab (flwr, flwr-datasets, datasets)...')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'flwr-datasets[vision]>=0.5.0', 'flwr'])
    import flwr_datasets
    import flwr
    print('✔ Dependências instaladas com sucesso!\n')

import os
import time
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import DirichletPartitioner, IidPartitioner
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Normalize, ToTensor

# Configurações de reprodutibilidade
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print(f"  🌸 INICIANDO APRENDIZADO FEDERADO ({estrategia_agregacao})")
print("=" * 70)
print(f"  • Estratégia de Agregação : {estrategia_agregacao}")
print(f"  • Rodadas Globais         : {num_rodadas}")
print(f"  • Clientes na Federação   : {num_clientes}")
print(f"  • Épocas Locais           : {epocas_locais}")
print(f"  • Taxa de Aprendizado (LR): {learning_rate}")
print(f"  • Batch Size Local        : {batch_size}")
print(f"  • Distribuição dos Dados  : {distribuicao_dados}")
print(f"  • Dispositivo de Cálculo  : {device}")
print("=" * 70 + "\n")

# Definição da Arquitetura CNN para CIFAR-10
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

def get_weights(net):
    return [val.cpu().numpy() for _, val in net.state_dict().items()]

def set_weights(net, parameters):
    params_dict = zip(net.state_dict().keys(), parameters)
    state_dict = {k: torch.tensor(v) for k, v in params_dict}
    net.load_state_dict(state_dict, strict=True)

# Pré-processamento CIFAR-10
_transforms = Compose([
    ToTensor(),
    Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

def _apply_transforms(batch):
    batch["img"] = [_transforms(img) for img in batch["img"]]
    return batch

print("[*] Baixando e particionando dataset CIFAR-10...")
if alpha_escolhido >= 100.0:
    partitioner = IidPartitioner(num_partitions=num_clientes)
else:
    partitioner = DirichletPartitioner(num_partitions=num_clientes, partition_by="label", alpha=alpha_escolhido, min_partition_size=10, self_balancing=True)
fds = FederatedDataset(dataset="uoft-cs/cifar10", partitioners={"train": partitioner})

test_ds = load_dataset("uoft-cs/cifar10", split="test").with_transform(_apply_transforms)
testloader = DataLoader(test_ds, batch_size=128, shuffle=False)

def train_client(net, trainloader, epochs, lr):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    net.train()
    total_loss, total, correct = 0.0, 0, 0
    for _ in range(epochs):
        for batch in trainloader:
            images, labels = batch["img"].to(device), batch["label"].to(device)
            optimizer.zero_grad()
            outputs = net(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return total_loss / max(total, 1), correct / max(total, 1)

def evaluate(net, testloader):
    criterion = nn.CrossEntropyLoss()
    net.eval()
    total_loss, total, correct = 0.0, 0, 0
    with torch.no_grad():
        for batch in testloader:
            images, labels = batch["img"].to(device), batch["label"].to(device)
            outputs = net(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return total_loss / max(total, 1), correct / max(total, 1)

# Inicializa o modelo global
global_net = Net().to(device)
init_loss, init_acc = evaluate(global_net, testloader)
print(f"\n▶ Acurácia Inicial (Sem Treinamento): {init_acc * 100:.2f}% | Loss: {init_loss:.4f}\n")

historico = []
tempo_inicio = time.time()

for r in range(1, num_rodadas + 1):
    r_start = time.time()
    client_weights = []
    client_sizes = []
    train_losses = []
    gw = get_weights(global_net)

    for cid in range(num_clientes):
        partition = fds.load_partition(cid, "train").with_transform(_apply_transforms)
        trainloader = DataLoader(partition, batch_size=batch_size, shuffle=True)
        local_net = Net().to(device)
        set_weights(local_net, gw)
        c_loss, _ = train_client(local_net, trainloader, epocas_locais, learning_rate)
        client_weights.append(get_weights(local_net))
        client_sizes.append(len(partition))
        train_losses.append(c_loss)

    # Agregação
    if estrategia_agregacao == "FedMedian":
        agg = []
        for l in range(len(client_weights[0])):
            stacked = np.stack([client_weights[c][l] for c in range(len(client_weights))], axis=0)
            agg.append(np.median(stacked, axis=0).astype(client_weights[0][l].dtype))
    else: # FedAvg
        total_s = sum(client_sizes)
        agg = []
        for l in range(len(client_weights[0])):
            l_sum = np.zeros_like(client_weights[0][l], dtype=np.float64)
            for c in range(len(client_weights)):
                l_sum += (client_sizes[c] / total_s) * client_weights[c][l].astype(np.float64)
            agg.append(l_sum.astype(client_weights[0][l].dtype))

    set_weights(global_net, agg)
    r_loss, r_acc = evaluate(global_net, testloader)
    r_dur = time.time() - r_start

    historico.append({"round": r, "acc": r_acc, "loss": r_loss, "train_loss": np.mean(train_losses), "time": r_dur})
    print(f"  [Rodada {r:02d}/{num_rodadas:02d}] Acurácia Global: {r_acc * 100:6.2f}% | Loss: {r_loss:.4f} | Duração: {r_dur:.1f}s")

tempo_total = time.time() - tempo_inicio
os.makedirs("resultados_flower", exist_ok=True)
torch.save(global_net.state_dict(), "resultados_flower/modelo_federado.pt")

print("\n" + "=" * 70)
print(f"  ✔ TREINAMENTO FEDERADO CONCLUÍDO COM SUCESSO EM {tempo_total:.1f}s!")
print(f"  • Acurácia Global Final : {historico[-1]['acc'] * 100:.2f}%")
print(f"  • Loss Global Final     : {historico[-1]['loss']:.4f}")
print("=" * 70)

## 4. Visualização dos Gráficos de Convergência
Exibe as curvas de Acurácia Global e de Perda (Loss) ao longo das rodadas de federação.

In [ ]:
rounds = [h["round"] for h in historico]
accs = [h["acc"] * 100 for h in historico]
losses = [h["loss"] for h in historico]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Gráfico de Acurácia
ax1.plot(rounds, accs, marker="o", color="#1976D2", lw=2.4, label="Acurácia Global (%)")
ax1.set_title("Convergência da Acurácia Global (CIFAR-10)", fontweight="bold", fontsize=12)
ax1.set_xlabel("Rodada Federada", fontsize=11)
ax1.set_ylabel("Acurácia (%)", fontsize=11)
ax1.set_ylim(0, 100)
ax1.grid(True, linestyle="--", alpha=0.3)
ax1.legend(loc="lower right")

# Gráfico de Loss
ax2.plot(rounds, losses, marker="s", color="#D32F2F", lw=2.4, label="Cross-Entropy Loss")
ax2.set_title("Evolução da Perda (Loss)", fontweight="bold", fontsize=12)
ax2.set_xlabel("Rodada Federada", fontsize=11)
ax2.set_ylabel("Loss", fontsize=11)
ax2.grid(True, linestyle="--", alpha=0.3)
ax2.legend(loc="upper right")

fig.suptitle(f"Curvas de Treinamento Federado — {estrategia_agregacao}", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()

grafico_path = "resultados_flower/curvas_convergencia.png"
fig.savefig(grafico_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✔ Gráfico salvo em: {grafico_path}")

## 5. Download dos Resultados (Sem Dependência de Google Drive)
Baixa o arquivo compactado `.zip` contendo os pesos treinados do modelo (`modelo_federado.pt`) e o gráfico de convergência diretamente para a pasta *Downloads* do seu computador.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/resultados_flower_puro", "zip", "resultados_flower")
print("[+] Compactação concluída: /content/resultados_flower_puro.zip")
print("[*] Iniciando download pelo navegador...")
files.download("/content/resultados_flower_puro.zip")